# GOAL: Build SARIMAX model

Note: using "previous day" would NOT mean that today is used to predict tomorrow. It means the prediction for today is used to predict tomorrow.

In [1]:
#import statements

#import everything!
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from PreRun import PreRun, PostRun
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX



#path to data
read_path = "../../../../data_ds_project/parquet_cleaned_energy"
#systems
good_systems_list = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1332, 4902, 4903]
reader_types = ["meter", "inverter", None]
#systems_cleaned
systems_cleaned = pd.read_csv("../../../data/core/systems_cleaned.csv")

In [ ]:
def use_sarimax(df_train, df_ho, p=0, d=0, q=0, P=0, D=0, Q=0, s=0):

    #separate out the exogenous variables
    df_train = df_train.set_index('time')
    df_ho = df_ho.set_index('time')

    # Remove duplicate indices (keep first occurrence)
    df_train = df_train[~df_train.index.duplicated(keep='first')]
    df_ho = df_ho[~df_ho.index.duplicated(keep='first')]

    df_train = df_train.asfreq('h')
    df_ho = df_ho.asfreq('h')

    energy_train = df_train['energy']
    #energy_ho = df_ho['energy']

    exog_train = df_train.drop(columns=['energy'])
    exog_ho = df_ho.drop(columns=['energy'])

    #fit the model
    model_sarimax = SARIMAX(energy_train, exog=exog_train, order=(p,d,q), seasonal_order = (P,D,Q,s)).fit(maxiter=300, disp=False)
    #predict
    y_pred = model_sarimax.forecast(len(df_ho), exog=exog_ho) 

    #error = PostRun.custom_error(energy_ho, y_pred, 1,2)

    return y_pred#, error

In [3]:
# system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]
# #system_reader_pairs = [(1283,'inverter'),(1283,'meter')]

# # p_choices =[2,3]
# # d_choices = [0,1]
# # q_choices = [0,1,2]

# # P_choices = [0,1,2,3]
# # D_choices = [0]
# # Q_choices = [0,1,2]

# p_choices =[2]
# d_choices = [1]
# q_choices = [0]

# P_choices = [0]
# D_choices = [0]
# Q_choices = [0]


# def process_system_pair(pair, p_choices, d_choices, q_choices, P_choices, D_choices, Q_choices, 
#                         read_path, systems_cleaned):
#     """Process a single system_reader_pair and return results"""
#     system_id = pair[0]
#     reader_type = pair[1]
#     print(f'starting system {system_id}, {reader_type}')

#     prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
#     prerun.load_data()
        
#     prerun.fill_missing_hours()
#     prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True)
#     print(prerun.amended_data)

#     prerun.add_weather_features_only()

#     all_data=prerun.amended_data

#     prerun.good_end_days_naive(10)
#     prerun.tts_of_data_using_end_days()

#     system_recorded_max = prerun.data['energy'].max()

#     # Build results as a dictionary to avoid pandas reindexing issues
#     all_errors_dict = {}
    
#     for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
#         print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
#         errors=[]
#         for pred_date in prerun.train_dates['date']:
#             #print(type(date))

        
#             train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
#                                     (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
#             ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
#                                 & (all_data['time'] <= pred_date)].reset_index(drop=True)

#             y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=0, D=D, Q=Q)
#             energy_ho = ho_data['energy']

#             #make sure value between 0 and highest observed max
#             y_pred = np.clip(y_pred, 0, system_recorded_max)
#             #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
#             darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int) # = 0 when both sunlight prop and irr are 0
#             y_pred = y_pred*np.array(darkness_mask)

#             error = PostRun.custom_error(energy_ho, y_pred, 1,2)

#             errors.append(error)

#         # Store error list in dictionary (avoids reindexing issues)
#         all_errors_dict[f'{p},{d},{q},{P},{D},{Q}'] = errors

#     # Convert dictionary to DataFrame at the end (single operation)
#     all_errors_df = pd.DataFrame(all_errors_dict)

#     # # Old code (commented out):
#     # all_errors_df = pd.DataFrame()
#     # 
#     # for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
#     #     print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
#     #     errors=[]
#     #     for pred_date in prerun.train_dates['date']:
#     #         train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
#     #                                 (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
#     #         ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
#     #                             & (all_data['time'] <= pred_date)].reset_index(drop=True)
#     #         y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=24, D=D, Q=Q)
#     #         energy_ho = ho_data['energy']
#     #         y_pred = np.clip(y_pred, 0, system_recorded_max)
#     #         darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int)
#     #         y_pred = y_pred*np.array(darkness_mask)
#     #         error = PostRun.custom_error(energy_ho, y_pred, 1,2)
#     #         errors.append(error)
#     #     all_errors_df[f'{p},{d},{q},{P},{D},{Q}'] = pd.Series(errors)

#     # Save results for this pair
#     output_path = Path(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors.csv')
#     output_path.parent.mkdir(parents=True, exist_ok=True)
#     all_errors_df.to_csv(output_path)
    
#     return (system_id, reader_type, all_errors_df)


# # # Serial execution (original code, commented out)
# # for pair in system_reader_pairs:
# #     system_id = pair[0]
# #     reader_type = pair[1]
# #     print(f'starting system {system_id}, {reader_type}')

# #     prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
# #     prerun.load_data()
# #         
# #     prerun.fill_missing_hours()
# #     prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True, highest_fourier_term_hour=2)
# #     print(prerun.amended_data)

# #     prerun.add_weather_features_only()

# #     all_data=prerun.amended_data

# #     prerun.good_end_days_naive(10)
# #     prerun.tts_of_data_using_end_days()

# #     system_recorded_max = prerun.data['energy'].max()

# #     all_errors_df = pd.DataFrame()
# #     
# #     for p,d,q,P,D,Q in product(p_choices,d_choices,q_choices,P_choices,D_choices,Q_choices):
# #         print(f'     starting p={p}, d={d}, q={q}, P={P}, D={D}, Q={Q}')
# #         errors=[]
# #         for pred_date in prerun.train_dates['date']:
# #             #print(type(date))

# #         
# #             train_data = all_data.loc[(all_data['time'] < pred_date - timedelta(days=1)) &
# #                                     (all_data['time'] >= pred_date - timedelta(days=10))].reset_index(drop=True)
# #             ho_data = all_data[(all_data['time'] >= pred_date - timedelta(days=1)) 
# #                                 & (all_data['time'] <= pred_date)].reset_index(drop=True)

# #             y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q, P=P, s=0, D=D, Q=Q)
# #             energy_ho = ho_data['energy']
# #             error = PostRun.custom_error(energy_ho, y_pred, 1,2)

# #             errors.append(error)

# #         all_errors_df[f'{p},{d},{q},{P},{D},{Q}'] = pd.Series(errors)

# # all_errors_df.to_csv(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors')


# # Parallel execution
# results = Parallel(n_jobs=-1)(
#     delayed(process_system_pair)(
#         pair, 
#         p_choices, 
#         d_choices, 
#         q_choices, 
#         P_choices, 
#         D_choices, 
#         Q_choices, 
#         read_path, 
#         systems_cleaned
#     ) 
#     for pair in system_reader_pairs
# )

# print("\nAll processing complete!")
# for system_id, reader_type, errors_df in results:
#     print(f'Completed: system {system_id}, {reader_type}')

In [13]:
system_reader_pairs = [(4,None),(10,None), (33, None), (50,None), (51,None), (1283,'inverter'),(1283,'meter')]

p_choices =[2,3]
d_choices = [0,1]
q_choices = [0,1]

for pair in system_reader_pairs:
    system_id = pair[0]
    reader_type = pair[1]
    print(f'starting system {system_id}, {reader_type}')

    prerun = PreRun(system_id = system_id, meter_or_inverter = reader_type, path=read_path, systems_cleaned=systems_cleaned)
    prerun.load_data()
        
    prerun.fill_missing_hours()
    prerun.add_energy_features_only(include_last_year = True, remove_last_year_nans=True, highest_fourier_term_hour=2)
    #print(prerun.amended_data)

    prerun.add_weather_features_only()

    all_data=prerun.amended_data

    prerun.good_end_days_naive(10)
    prerun.tts_of_data_using_end_days()

    system_recorded_max = prerun.data['energy'].max()

    #get all train/ho data sets ONCE so it's not redone for each (p,d,q)
    splits = []

    for pred_date in prerun.train_dates['date']:
        train_mask = (
            (all_data['time'] < pred_date - timedelta(days=1)) &
            (all_data['time'] >= pred_date - timedelta(days=10))
        )
        ho_mask = (
            (all_data['time'] >= pred_date - timedelta(days=1)) &
            (all_data['time'] < pred_date + timedelta(days=1))
        )

        train_data = all_data.loc[train_mask].reset_index(drop=True)
        ho_data = all_data.loc[ho_mask].reset_index(drop=True)

        splits.append((pred_date, train_data, ho_data))
    
    results_dict = {}
    # print(system_recorded_max)
    # print(ho_data)
    # print(ho_data.columns)

    for p,d,q in product(p_choices,d_choices,q_choices):
        print(f'     starting p={p}, d={d}, q={q}')
        errors=[]
        
        for pred_date, train_data, ho_data in splits:
            try: 
                y_pred = use_sarimax(train_data, ho_data, p=p, d=d, q=q)
                # print(f"original y_pred: {y_pred}")

                #make sure value between 0 and highest observed max
                y_pred = np.clip(y_pred, 0, system_recorded_max)
                #make sure that if it's not sunlight hours and irradiance = 0, then energy = 0
                darkness_mask = (~((ho_data['proportion_daytime']==0) & (ho_data['global_tilted_irradiance']==0))).astype(int) # = 0 when both sublight prop and irr are 0
                y_pred = y_pred*np.array(darkness_mask)
                # print(f"y_pred with insurance: {y_pred}")

                energy_ho = ho_data['energy']
                # print(f"energy_ho: {energy_ho}")
                
                error = PostRun.custom_error(energy_ho.iloc[24:], y_pred.iloc[24:], 1,2)
                # print(f'error = {error}')
            except Exception as e:
                print(f'FAILED: (p,d,q) = {p,d,q} --> {type(e).__name__}: {e}')
                error = -1
                

            errors.append(error)

        results_dict[f"{p},{d},{q}"] = errors
    all_errors_df = pd.DataFrame(results_dict)
    all_errors_df.to_csv(f'sarimax_errors/{system_id}_{reader_type}_sarimax_errors.csv', index = False)

starting system 4, None
     starting p=2, d=0, q=0


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=2, d=0, q=1


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-inv

     starting p=2, d=1, q=0


...

c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=3, d=1, q=0


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

     starting p=3, d=1, q=1


c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\hjpha\anaconda3\envs\erdos_ds_environment\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed t

In [ ]:
#compare hyperparameters
#System 4
print('System 4, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/4_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 4, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#all seem to be fine. Leaning toward 3,0,1. Then 3,0,0. Then 2,0,0

System 4, None, SARIMAX
recorded system max: 1.0241429033333334
  Hyperparameters: 2,0,0
     mean: 0.012058420364281322, median: 0.005799299106815649, min: 0.0001786873605232, max: 0.0984827482374984, std: 0.01635586775971226
  Hyperparameters: 2,0,1
     mean: 0.012522697268532395, median: 0.00550527394669135, min: 0.0001677720843443, max: 0.0987941805915381, std: 0.0169409617552593
  Hyperparameters: 2,1,0
     mean: 0.013711235610101586, median: 0.00686699271542355, min: 0.0006492290227219, max: 0.1028747279225662, std: 0.01800034748559317
  Hyperparameters: 2,1,1
     mean: 0.014283685937524318, median: 0.006944147513717551, min: 0.000658252153881, max: 0.1097374409647573, std: 0.018459197602437808
  Hyperparameters: 3,0,0
     mean: 0.01206599043219728, median: 0.00579062094976795, min: 0.0001913633346663, max: 0.0987620640271799, std: 0.016355566085032528
  Hyperparameters: 3,0,1
     mean: 0.012308429772542463, median: 0.005793385501036049, min: 0.0001898282294371, max: 0.09879

In [ ]:
#compare hyperparameters
#System 10
print('System 10, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/10_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 10, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#most fine. max's are a bit high.
#2,0,0; 3,0,0; 2,0,1; 3,0,1

System 10, None, SARIMAX
recorded system max: 1.185825
  Hyperparameters: 2,0,0
     mean: 0.020585519863911633, median: 0.0072937734963689, min: 0.0002591027718161, max: 0.1332870291469238, std: 0.03137582073135409
  Hyperparameters: 2,0,1
     mean: 0.02094365705218247, median: 0.0072861311098557, min: 0.000239998216452, max: 0.1357527918507043, std: 0.03237807178709216
  Hyperparameters: 2,1,0
     mean: 0.02622466048793849, median: 0.0124841585601312, min: 0.0007589157096734, max: 0.1386062195039717, std: 0.03356645553611215
  Hyperparameters: 2,1,1
     mean: 0.02672550803166853, median: 0.0128162209960295, min: 0.0011428231441642, max: 0.1500857825323143, std: 0.035420185927427586
  Hyperparameters: 3,0,0
     mean: 0.020697937762777084, median: 0.0073162455204369, min: 0.0002397735075866, max: 0.1357022721874045, std: 0.0317107103469593
  Hyperparameters: 3,0,1
     mean: 0.021054921436047692, median: 0.0072843546851442, min: 0.000240000265541, max: 0.1357572256981232, std: 0.03

In [ ]:
#compare hyperparameters
#System 33
print('System 33, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/33_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 33, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#several ok ones. Max's are getting higher though.
#3,0,0; 2,0,0; 3,0,1; 2,0,1

System 33, None, SARIMAX
recorded system max: 2.4044553333333334
  Hyperparameters: 2,0,0
     mean: 0.06057244418404123, median: 0.0415561690908592, min: 0.0025820549436002, max: 0.5340349569696848, std: 0.06959925439959738
  Hyperparameters: 2,0,1
     mean: 0.06144616163543015, median: 0.0416417181295475, min: 0.0025073874865838, max: 0.5340204616451617, std: 0.0703781105552384
  Hyperparameters: 2,1,0
     mean: 0.08031348563360503, median: 0.0483680142958236, min: 0.0050052885424658, max: 0.8823798798488179, std: 0.10387120029352909
  Hyperparameters: 2,1,1
     mean: 0.07741388753156135, median: 0.0487853809854863, min: 0.0043488407495522, max: 0.7754733113381765, std: 0.09075706481447585
  Hyperparameters: 3,0,0
     mean: 0.060492803718607505, median: 0.0414310681208748, min: 0.0025648388701059, max: 0.5340059701703418, std: 0.06942677904037918
  Hyperparameters: 3,0,1
     mean: 0.060983653666756246, median: 0.0414455287940991, min: 0.0025683255170717, max: 0.5340163727732725,

In [ ]:
#compare hyperparameters
#System 50
print('System 50, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/50_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 50, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#3,0,0; 3,0,1; 2,0,0; 2,0,1

System 50, None, SARIMAX
recorded system max: 7.072975
  Hyperparameters: 2,0,0
     mean: 0.42744412452266134, median: 0.2447224093483765, min: 9.573573035474448e-12, max: 4.65270696866, std: 0.5816764570761119
  Hyperparameters: 2,0,1
     mean: 0.436378339846918, median: 0.25262891691158207, min: 9.573573035099763e-12, max: 4.634019304346637, std: 0.5878911497022081
  Hyperparameters: 2,1,0
     mean: 0.5067549001326345, median: 0.3086088358634559, min: 9.151976877247198e-12, max: 3.862061325698107, std: 0.6307915337919323
  Hyperparameters: 2,1,1
     mean: 0.5003265645601842, median: 0.3002413832059777, min: -1.0, max: 4.647389839730853, std: 0.672973433419761
  Hyperparameters: 3,0,0
     mean: 0.4230976045157921, median: 0.2276994828286624, min: 9.573573035390575e-12, max: 4.626920578829719, std: 0.5785685804683197
  Hyperparameters: 3,0,1
     mean: 0.4195322094807093, median: 0.22226530819927076, min: 9.573573035600666e-12, max: 4.636143632384246, std: 0.5797324753609758
  Hyp

In [ ]:
#compare hyperparameters
#System 51
print('System 51, None, SARIMAX')
errors = pd.read_csv('sarimax_errors/51_None_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 51, meter_or_inverter = None, path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )

#much less clear. There are some clear "lowest mean", but some others seem to have a much lower spread.
# Preferred because of mean: 3,0,0; 3,0,1 (either); followed by 2,0,0 and 2,0,1 (either order)
# but 2,1,0 seems to be the most "bunched together", despite the high mean

System 51, None, SARIMAX
recorded system max: 7.2368749999999995
  Hyperparameters: 2,0,0
     mean: 0.527453466245032, median: 0.28143721664585825, min: 0.0116106245750907, max: 4.842273684864219, std: 0.7252155683132562
  Hyperparameters: 2,0,1
     mean: 0.5334692655937507, median: 0.2826064440948759, min: 0.0220002576337766, max: 4.815697303376325, std: 0.7182889634990434
  Hyperparameters: 2,1,0
     mean: 0.5979284172882094, median: 0.34800719572278027, min: 0.0243773084006721, max: 3.710808632583363, std: 0.7041900538634319
  Hyperparameters: 2,1,1
     mean: 0.6066886683607097, median: 0.3382429444361069, min: 0.0319722419700745, max: 4.74407324170025, std: 0.7741404436389625
  Hyperparameters: 3,0,0
     mean: 0.517999134291719, median: 0.27683963847577076, min: 0.0118955974833283, max: 4.80576169073969, std: 0.7224291146787446
  Hyperparameters: 3,0,1
     mean: 0.5171412294243217, median: 0.27722801347940934, min: 0.0118847999594964, max: 4.81746506312448, std: 0.72469961414

In [ ]:
#compare hyperparameters
#System 1283
print('System 1283, Inverter, SARIMAX')
errors = pd.read_csv('sarimax_errors/1283_inverter_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'inverter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#again the main 4 are really similar. 2,0,0/3,0,0 have slightly lower means. 2,0,1/3,0,1 have slightly lower max's and std

System 1283, Inverter, SARIMAX
recorded system max: 214.57368458333335
  Hyperparameters: 2,0,0
     mean: 229.1040433525471, median: 123.92463534881584, min: 2.8894316643246194e-05, max: 1726.7680502554022, std: 349.05215083924054
  Hyperparameters: 2,0,1
     mean: 230.8756312056124, median: 120.67760022462717, min: 2.0444422911909248e-05, max: 1720.2890793065178, std: 348.33405156954603
  Hyperparameters: 2,1,0
     mean: 296.0285924177304, median: 108.8733042763944, min: 0.0001011260236305, max: 2925.557993437953, std: 526.0964145712001
  Hyperparameters: 2,1,1
     mean: 280.2638870687968, median: 121.62392176366443, min: 0.0001021189176155, max: 2290.8733370834257, std: 471.66160048979987
  Hyperparameters: 3,0,0
     mean: 229.92742726698648, median: 123.33730912571295, min: 3.028366901563785e-05, max: 1725.680425559364, std: 349.1933680788236
  Hyperparameters: 3,0,1
     mean: 230.7668480873405, median: 119.80770859223816, min: 2.998278703437863e-05, max: 1724.2234310657914, s

In [ ]:
#compare hyperparameters
#System 1283
print('System 1283, Meter, SARIMAX')
errors = pd.read_csv('sarimax_errors/1283_meter_sarimax_errors.csv')
hyperparams = errors.columns

prerun = PreRun(system_id = 1283, meter_or_inverter = 'meter', path=read_path, systems_cleaned=systems_cleaned)
prerun.load_data()
system_recorded_max = prerun.data['energy'].max()
print(f'recorded system max: {system_recorded_max}')

for hp in hyperparams:
    err = errors[hp]
    print(f'  Hyperparameters: {hp}')
    print(f'     mean: {err.mean()}, median: {err.median()}, min: {err.min()}, max: {err.max()}, std: {err.std()}' )
    
#again the main 4 are really similar.
# 3,0,0; 2,0,1; 2,0,0; 3,0,1

System 1283, Meter, SARIMAX
recorded system max: 426.25958333333335
  Hyperparameters: 2,0,0
     mean: 943.2232742994668, median: 411.5276146804224, min: 0.0001283838156566, max: 6858.265307843451, std: 1464.3395725587538
  Hyperparameters: 2,0,1
     mean: 956.9730635594614, median: 384.7903607007052, min: 7.583638937490035e-05, max: 6838.103980236051, std: 1465.6980094013336
  Hyperparameters: 2,1,0
     mean: 1243.7199158536785, median: 381.0036839435041, min: 0.0001659221845717, max: 11980.303564204298, std: 2126.470254376208
  Hyperparameters: 2,1,1
     mean: 1114.1954638691896, median: 370.1762297457271, min: 0.0001644655700127, max: 9772.36119033242, std: 1888.8817993924692
  Hyperparameters: 3,0,0
     mean: 945.6136355403177, median: 455.182400712996, min: 0.0001323027417752, max: 6845.829637824889, std: 1461.6465216867355
  Hyperparameters: 3,0,1
     mean: 963.0196893370195, median: 398.2300714663497, min: 0.0001322217972062, max: 6841.304606510438, std: 1463.3722140266725